In [ ]:
"""
Paired Bootstrap per il confronto tra 2 modelli.
 
Scenario:
  - N record, ciascuno con uno score da modello A e modello B
  - Si vuole verificare se la differenza di performance è significativa
 
Include:
  1. Intervallo di confidenza (percentile e BCa)
  2. Test d'ipotesi con bootstrap sotto H0
  3. Visualizzazione dei risultati
"""

In [1]:

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [2]:
# ──────────────────────────────────────────────
# 1. Dati simulati
# ──────────────────────────────────────────────
np.random.seed(42)
N = 500  # numero di record
 
# Simula score per-record (es. errore assoluto, log-loss, ecc.)
# Modello A: leggermente migliore (score più basso = meglio)
scores_a = np.random.beta(2, 5, size=N)       # media ~0.286
scores_b = np.random.beta(2, 4.5, size=N)     # media ~0.308
 
# Differenze per-record (A - B): valori negativi = A migliore
d = scores_a - scores_b
d_mean_obs = d.mean()
 
print("=" * 60)
print("DATI OSSERVATI")
print("=" * 60)
print(f"  N record:              {N}")
print(f"  Media score A:         {scores_a.mean():.4f}")
print(f"  Media score B:         {scores_b.mean():.4f}")
print(f"  Differenza media (A-B): {d_mean_obs:.4f}")
print(f"  Std differenze:        {d.std(ddof=1):.4f}")

DATI OSSERVATI
  N record:              500
  Media score A:         0.2870
  Media score B:         0.3062
  Differenza media (A-B): -0.0192
  Std differenze:        0.2239


In [19]:
# ──────────────────────────────────────────────
# 2. Bootstrap: Intervallo di confidenza
# ──────────────────────────────────────────────
B = 10_000  # iterazioni bootstrap
alpha = 0.05
 
rng = np.random.default_rng(42)
 
# Ricampiona gli INDICI (preserva l'accoppiamento)
boot_diffs = np.empty(B)
for b in range(B):
    idx = rng.integers(0, N, size=N)
    boot_diffs[b] = scores_a[idx].mean() - scores_b[idx].mean()
  
# --- Intervallo percentile semplice ---
ci_lo_pct = np.percentile(boot_diffs, 100 * alpha / 2)
ci_hi_pct = np.percentile(boot_diffs, 100 * (1 - alpha / 2))
 
# --- Intervallo BCa (Bias-Corrected and Accelerated) ---
# Correzione per il bias
z0 = stats.norm.ppf(np.mean(boot_diffs < d_mean_obs))
 
# Accelerazione (jackknife)
jackknife_means = np.empty(N)  # N leave-one-out means
for i in range(N):
    mask = np.ones(N, dtype=bool)  # [True, True, ..., True]
    # Esclude elemento i e calcola la media sui restanti
    mask[i] = False
    jackknife_means[i] = d[mask].mean()
 
jk_mean = jackknife_means.mean()
num = np.sum((jk_mean - jackknife_means) ** 3)
den = 6 * (np.sum((jk_mean - jackknife_means) ** 2) ** 1.5)
a_hat = num / den if den != 0 else 0
 
# Quantili corretti
z_alpha_lo = stats.norm.ppf(alpha / 2)
z_alpha_hi = stats.norm.ppf(1 - alpha / 2)
 
p_lo = stats.norm.cdf(z0 + (z0 + z_alpha_lo) / (1 - a_hat * (z0 + z_alpha_lo)))
p_hi = stats.norm.cdf(z0 + (z0 + z_alpha_hi) / (1 - a_hat * (z0 + z_alpha_hi)))
 
ci_lo_bca = np.percentile(boot_diffs, 100 * p_lo)
ci_hi_bca = np.percentile(boot_diffs, 100 * p_hi)
 
print()
print("=" * 60)
print(f"INTERVALLI DI CONFIDENZA AL {100*(1-alpha):.0f}%")
print("=" * 60)
print(f"  Percentile:  [{ci_lo_pct:.4f}, {ci_hi_pct:.4f}]")
print(f"  BCa:         [{ci_lo_bca:.4f}, {ci_hi_bca:.4f}]")
print(f"  Lo zero è {'FUORI' if ci_hi_bca < 0 or ci_lo_bca > 0 else 'DENTRO'} l'intervallo BCa")


INTERVALLI DI CONFIDENZA AL 95%
  Percentile:  [-0.0391, 0.0003]
  BCa:         [-0.0394, 0.0001]
  Lo zero è DENTRO l'intervallo BCa


In [ ]:
# ──────────────────────────────────────────────
# 3. Test d'ipotesi con bootstrap sotto H0
# ──────────────────────────────────────────────
# Centramento: forza H0 (media differenze = 0)
d_centered = d - d_mean_obs  # ora ha media 0
 
T_obs = np.abs(d_mean_obs)   # statistica test osservata (two-sided)
 
boot_T_h0 = np.empty(B)
for b in range(B):
    idx = rng.integers(0, N, size=N)
    boot_T_h0[b] = np.abs(d_centered[idx].mean())  # Media delle differenze centrate

# p-value: proporzione di T sotto H0 >= T osservato
p_value = np.mean(boot_T_h0 >= T_obs)
 
print()
print("=" * 60)
print("TEST D'IPOTESI BOOTSTRAP (two-sided)")
print("=" * 60)
print(f"  H0: E[score_A - score_B] = 0")
print(f"  Statistica osservata |d̄|: {T_obs:.4f}")
print(f"  p-value bootstrap:        {p_value:.4f}")
print(f"  Conclusione (α=0.05):     {'Rifiuta H0' if p_value < 0.05 else 'Non rifiuta H0'}")
 
# Confronto con test classici
_, p_paired_t = stats.ttest_rel(scores_a, scores_b)
_, p_wilcoxon = stats.wilcoxon(d)
 
print()
print("  --- Confronto con test classici ---")
print(f"  Paired t-test p-value:    {p_paired_t:.4f}")
print(f"  Wilcoxon signed-rank:     {p_wilcoxon:.4f}")


TEST D'IPOTESI BOOTSTRAP (two-sided)
  H0: E[score_A - score_B] = 0
  Statistica osservata |d̄|: 0.0192
  p-value bootstrap:        0.0563
  Conclusione (α=0.05):     Non rifiuta H0

  --- Confronto con test classici ---
  Paired t-test p-value:    0.0554
  Wilcoxon signed-rank:     0.0883


In [46]:
r_plus_mask = d > 0
r_minus_mask = d < 0

r_plus = np.sum(r_plus_mask)
r_minus = np.sum(r_minus_mask)

T = min(r_plus, r_minus)

In [47]:
num = T - (0.25 * N * (N + 1))
den = (N * (N + 1) * (2*N + 1) / 24)**0.5
z = num / den

In [48]:
z

np.float64(-19.302812184554533)

In [49]:
N

500